# Train Forecasting Model

## Imports

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error

import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

import joblib
from watchdog.observers.winapi import read_events

## Loading Data

In [2]:
PROCESSED_DATA_DIR= '../data/processed'
MODELS_DIR= '../models'

In [3]:
data_path= os.path.join(PROCESSED_DATA_DIR, 'forecasting_training_data.csv')

In [4]:
df= pd.read_csv(filepath_or_buffer= data_path)

In [5]:
df.head()

,sale_date,category,units_sold,daily_revenue
0,2016-09-15,health_beauty,3,134.97
1,2016-10-03,fashion_shoes,1,29.99
2,2016-10-03,furniture_decor,2,194.80
3,2016-10-03,sports_leisure,2,58.39
4,2016-10-03,toys,1,128.90


In [6]:
df.describe()

,units_sold,daily_revenue
count,18311.000000,18311.000000
mean,5.932936,712.442297
std,7.285815,969.197817
min,1.000000,3.850000
25%,1.000000,119.730000
50%,3.000000,349.900000
75%,8.000000,923.335000
max,192.000000,17667.020000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18311 entries, 0 to 18310
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sale_date      18311 non-null  object 
 1   category       18311 non-null  object 
 2   units_sold     18311 non-null  int64  
 3   daily_revenue  18311 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 572.3+ KB


In [8]:
df.shape

(18311, 4)

## Feature Engineering

In [9]:
# Converting sale_date to DateTime:
df['sale_date'] = pd.to_datetime(df['sale_date'])

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18311 entries, 0 to 18310
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   sale_date      18311 non-null  datetime64[ns]
 1   category       18311 non-null  object        
 2   units_sold     18311 non-null  int64         
 3   daily_revenue  18311 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 572.3+ KB


In [11]:
# Re-Sampling to Weekly Buckets by Category:
weekly_df= df.groupby('category').resample('W', on='sale_date').agg({
    'daily_revenue':'sum',
    'units_sold': 'sum'
}).reset_index()

In [12]:
weekly_df.head()

,category,sale_date,daily_revenue,units_sold
0,agro_industry_and_commerce,2017-01-29,43.98,2
1,agro_industry_and_commerce,2017-02-05,43.98,2
2,agro_industry_and_commerce,2017-02-12,114.89,2
3,agro_industry_and_commerce,2017-02-19,65.97,3
4,agro_industry_and_commerce,2017-02-26,21.99,1


In [13]:
# Renaming Columns:
weekly_df.rename(columns={
    'sale_date': 'week_ending_date',
    'daily_revenue': 'weekly_revenue',
    'units_sold': 'weekly_units'
}, inplace= True)

In [14]:
weekly_df.head()

,category,week_ending_date,weekly_revenue,weekly_units
0,agro_industry_and_commerce,2017-01-29,43.98,2
1,agro_industry_and_commerce,2017-02-05,43.98,2
2,agro_industry_and_commerce,2017-02-12,114.89,2
3,agro_industry_and_commerce,2017-02-19,65.97,3
4,agro_industry_and_commerce,2017-02-26,21.99,1


In [15]:
# Extracting Calendar Features:
weekly_df['year']= weekly_df['week_ending_date'].dt.isocalendar().year
weekly_df['week_of_year']= weekly_df['week_ending_date'].dt.isocalendar().week

In [16]:
weekly_df.head()

,category,week_ending_date,weekly_revenue,weekly_units,year,week_of_year
0,agro_industry_and_commerce,2017-01-29,43.98,2,2017,4
1,agro_industry_and_commerce,2017-02-05,43.98,2,2017,5
2,agro_industry_and_commerce,2017-02-12,114.89,2,2017,6
3,agro_industry_and_commerce,2017-02-19,65.97,3,2017,7
4,agro_industry_and_commerce,2017-02-26,21.99,1,2017,8


In [17]:
# Adding Auto-Regressive / LAG Features:
# 1. Last week:
weekly_df['lag_1_revenue']= weekly_df.groupby('category')['weekly_revenue'].shift(1)
weekly_df['lag_1_units']= weekly_df.groupby('category')['weekly_units'].shift(1)

In [18]:
weekly_df.head()

,category,week_ending_date,weekly_revenue,weekly_units,year,week_of_year,lag_1_revenue,lag_1_units
0,agro_industry_and_commerce,2017-01-29,43.98,2,2017,4,NaN,NaN
1,agro_industry_and_commerce,2017-02-05,43.98,2,2017,5,43.98,2.0
2,agro_industry_and_commerce,2017-02-12,114.89,2,2017,6,43.98,2.0
3,agro_industry_and_commerce,2017-02-19,65.97,3,2017,7,114.89,2.0
4,agro_industry_and_commerce,2017-02-26,21.99,1,2017,8,65.97,3.0


In [19]:
# EWMA Prioritizing last 4 Weeks:
weekly_df['ewma_4_revenue']= weekly_df.groupby('category')['weekly_revenue'].transform(
    lambda x: x.ewm(span= 4, adjust=False).mean().shift(1)
)

weekly_df['ewma_4_units']= weekly_df.groupby('category')['weekly_units'].transform(
    lambda x: x.ewm(span= 4, adjust=False).mean().shift(1)
)

In [20]:
weekly_df.head()

,category,week_ending_date,weekly_revenue,weekly_units,year,week_of_year,lag_1_revenue,lag_1_units,ewma_4_revenue,ewma_4_units
0,agro_industry_and_commerce,2017-01-29,43.98,2,2017,4,NaN,NaN,NaN,NaN
1,agro_industry_and_commerce,2017-02-05,43.98,2,2017,5,43.98,2.0,43.9800,2.0
2,agro_industry_and_commerce,2017-02-12,114.89,2,2017,6,43.98,2.0,43.9800,2.0
3,agro_industry_and_commerce,2017-02-19,65.97,3,2017,7,114.89,2.0,72.3440,2.0
4,agro_industry_and_commerce,2017-02-26,21.99,1,2017,8,65.97,3.0,69.7944,2.4


In [21]:
# Dropping Null Values Created by LAG Features:
weekly_df= weekly_df.dropna().reset_index(drop= True)
weekly_df.head()

,category,week_ending_date,weekly_revenue,weekly_units,year,week_of_year,lag_1_revenue,lag_1_units,ewma_4_revenue,ewma_4_units
0,agro_industry_and_commerce,2017-02-05,43.98,2,2017,5,43.98,2.0,43.98000,2.00
1,agro_industry_and_commerce,2017-02-12,114.89,2,2017,6,43.98,2.0,43.98000,2.00
2,agro_industry_and_commerce,2017-02-19,65.97,3,2017,7,114.89,2.0,72.34400,2.00
3,agro_industry_and_commerce,2017-02-26,21.99,1,2017,8,65.97,3.0,69.79440,2.40
4,agro_industry_and_commerce,2017-03-05,0.00,0,2017,9,21.99,1.0,50.67264,1.84


In [22]:
weekly_df.shape

(5919, 10)

## Train Test Split

In [23]:
# Cutoff Date for Last 8 weeks:
cutoff_date= weekly_df['week_ending_date'].max() - pd.Timedelta(weeks= 8)

# Training Data:
train_df= weekly_df[weekly_df['week_ending_date'] < cutoff_date]
test_df= weekly_df[weekly_df['week_ending_date'] >= cutoff_date]

In [24]:
print(train_df.shape)
print(test_df.shape)

(5387, 10)
(532, 10)


## Data Pre-Processing

In [25]:
# Base Features:
categorical_features= ['category']
base_numeric_features= ['year', 'week_of_year']

In [26]:
# Preprocessor:
preprocessor= ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        # Will add numeric features dynamically based on the model
    ],
    remainder='passthrough'
)

## Revenue Model

In [28]:
# Features and Target for Revenue Model:
revenue_features= ['category', 'year', 'week_of_year', 'lag_1_revenue', 'ewma_4_revenue']
revenue_target= 'weekly_revenue'

In [29]:
# Training And Test Dataset:
X_train_rev, y_train_rev= train_df[revenue_features], train_df[revenue_target]
X_test_rev, y_test_rev= test_df[revenue_features], test_df[revenue_target]

In [30]:
print(X_train_rev.shape)
print(y_train_rev.shape)
print(X_test_rev.shape)
print(y_test_rev.shape)

(5387, 5)
(5387,)
(532, 5)
(532,)


In [31]:
# Pre-Processing and Model Fitting Pipeline:
revenue_pipeline= Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('scaler', StandardScaler(with_mean= False)),
        ('regressor', xgb.XGBRegressor(n_estimators= 200,
                                       learning_rate= 0.05,
                                       max_depth= 5,
                                       random_state= 42))
    ]
)

In [32]:
# Model Fitting:
revenue_pipeline.fit(X_train_rev, y_train_rev)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['category'])])),
                ('scaler', StandardScaler(with_mean=False)),
                ('regressor',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=None, device=None,
                              early_s...
                              feature_types=None, gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=5, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=200, n_jobs=None,
                              num_parallel_tree=None, random_state=42, ...))])

In [33]:
# Evaluating Revenue Model on Test Data:
revenue_preds= revenue_pipeline.predict(X_test_rev)

In [34]:
print(f"REVENUE MODEL - Mean Absolute Error: ${mean_absolute_error(y_test_rev, revenue_preds):.2f}")

print(f"REVENUE MODEL - Root Mean Squared Error: ${np.sqrt(mean_squared_error(y_test_rev, revenue_preds)):.2f}")

REVENUE MODEL - Mean Absolute Error: $1465.55
REVENUE MODEL - Root Mean Squared Error: $2761.32


## Inventory Model

In [35]:
# Features and Target for Inventory Model:
inventory_features= ['category', 'year', 'week_of_year', 'lag_1_units', 'ewma_4_units']
inventory_target= 'weekly_units'

In [36]:
# Training and Test Dataset:
X_train_inv, y_train_inv= train_df[inventory_features], train_df[inventory_target]
X_test_inv, y_test_inv= test_df[inventory_features], test_df[inventory_target]

In [37]:
print(X_train_inv.shape)
print(X_test_inv.shape)
print(y_train_inv.shape)
print(y_test_inv.shape)

(5387, 5)
(532, 5)
(5387,)
(532,)


In [38]:
# Pre-Processing and Model Fitting Pipeline:
inventory_pipeline= Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('scaler', StandardScaler(with_mean= False)),
        ('regressor', xgb.XGBRegressor(n_estimators= 200,
                                       learning_rate= 0.05,
                                       max_depth= 5,
                                       random_state= 42))
    ]
)

In [39]:
# Fitting The Pipeline:
inventory_pipeline.fit(X_train_inv, y_train_inv)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['category'])])),
                ('scaler', StandardScaler(with_mean=False)),
                ('regressor',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=None, device=None,
                              early_s...
                              feature_types=None, gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=5, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=200, n_jobs=None,
                              num_parallel_tree=None, random_state=42, ...))])

In [40]:
# Evaluating Inventory Model on Test Data:
inventory_preds= inventory_pipeline.predict(X_test_inv)

In [44]:
print(f"Inventory MODEL - Mean Absolute Error: {mean_absolute_error(y_test_inv, inventory_preds):.2f}")

print(f"Inventory MODEL - Root Mean Squared Error: {np.sqrt(mean_squared_error(y_test_inv, inventory_preds)):.2f}")

Inventory MODEL - Mean Absolute Error: 10.15
Inventory MODEL - Root Mean Squared Error: 19.99


In [45]:
weekly_df['weekly_units'].mean()

18.302078053725293

## Checking Error Rate of Both Models for Top 5 selling Categories

### Revenue Model

In [46]:
# Create a dataframe of just the test results
results_df = X_test_rev.copy()
results_df['actual_revenue'] = y_test_rev
results_df['predicted_revenue'] = revenue_preds

# Find the top 5 largest categories by total revenue in the test set
top_categories = results_df.groupby('category')['actual_revenue'].sum().nlargest(5).index

print("--- EVALUATION FOR TOP 5 CATEGORIES ---")
for cat in top_categories:
    cat_data = results_df[results_df['category'] == cat]

    cat_mean = cat_data['actual_revenue'].mean()
    cat_mae = mean_absolute_error(cat_data['actual_revenue'], cat_data['predicted_revenue'])

    # Calculate the relative error percentage
    if cat_mean > 0:
        error_pct = (cat_mae / cat_mean) * 100
    else:
        error_pct = 0

    print(f"Category: {cat}")
    print(f"  Avg Weekly Revenue: ${cat_mean:.2f}")
    print(f"  Mean Absolute Error: ${cat_mae:.2f}")
    print(f"  Relative Error Rate: {error_pct:.1f}%\n")

--- EVALUATION FOR TOP 5 CATEGORIES ---
Category: health_beauty
  Avg Weekly Revenue: $24670.90
  Mean Absolute Error: $7852.60
  Relative Error Rate: 31.8%

Category: watches_gifts
  Avg Weekly Revenue: $17739.97
  Mean Absolute Error: $6065.46
  Relative Error Rate: 34.2%

Category: housewares
  Avg Weekly Revenue: $13239.68
  Mean Absolute Error: $5540.27
  Relative Error Rate: 41.8%

Category: bed_bath_table
  Avg Weekly Revenue: $12671.62
  Mean Absolute Error: $5190.06
  Relative Error Rate: 41.0%

Category: sports_leisure
  Avg Weekly Revenue: $11437.60
  Mean Absolute Error: $4307.79
  Relative Error Rate: 37.7%



### Inventory Model

In [48]:
# Create a dataframe of just the test results
results_df = X_test_inv.copy()
results_df['actual_units'] = y_test_inv
results_df['predicted_units'] = inventory_preds

# Find the top 5 largest categories by total units sold in the test set
top_categories = results_df.groupby('category')['actual_units'].sum().nlargest(5).index

print("--- EVALUATION FOR TOP 5 CATEGORIES ---")
for cat in top_categories:
    cat_data = results_df[results_df['category'] == cat]

    cat_mean = cat_data['actual_units'].mean()
    cat_mae = mean_absolute_error(cat_data['actual_units'], cat_data['predicted_units'])

    # Calculate the relative error percentage
    if cat_mean > 0:
        error_pct = (cat_mae / cat_mean) * 100
    else:
        error_pct = 0

    print(f"Category: {cat}")
    print(f"  Avg Weekly units Sold: {cat_mean:.2f}")
    print(f"  Mean Absolute Error: {cat_mae:.2f}")
    print(f"  Relative Error Rate: {error_pct:.1f}%\n")

--- EVALUATION FOR TOP 5 CATEGORIES ---
Category: health_beauty
  Avg Weekly units Sold: 177.22
  Mean Absolute Error: 54.11
  Relative Error Rate: 30.5%

Category: bed_bath_table
  Avg Weekly units Sold: 139.22
  Mean Absolute Error: 53.27
  Relative Error Rate: 38.3%

Category: housewares
  Avg Weekly units Sold: 128.56
  Mean Absolute Error: 46.16
  Relative Error Rate: 35.9%

Category: sports_leisure
  Avg Weekly units Sold: 102.44
  Mean Absolute Error: 33.36
  Relative Error Rate: 32.6%

Category: watches_gifts
  Avg Weekly units Sold: 101.78
  Mean Absolute Error: 31.06
  Relative Error Rate: 30.5%

